# Topic modelling for SEO with LDA (2026 refresh)

Companion notebook for the MLforSEO guide **"Topic Modelling for SEO with LDA (Latent Dirichlet Allocation)"** by Lazarina Stoy.

What you get, top to bottom:

1. Load your own content (a crawl export) or the demo dataset
2. Clean and tokenise the text (stopwords, bigrams, topic-general words)
3. A word cloud of the cleaned corpus
4. Train an LDA model with **gensim 4.4**
5. Judge it: **perplexity** and **coherence (c_v)**, then tune the number of topics, alpha and eta
6. The interactive **pyLDAvis** map
7. Per-page topic probabilities (soft membership), top keywords per topic, top pages per topic
8. Topic-to-topic correlations and **internal-link candidates** (the same outputs as the no-code jsLDA app, at scale)
9. **NMF** as a comparison model
10. A **short-text experiment**: the same model on titles vs full text, to see why LDA struggles on short text
11. CSV exports

Runs in Google Colab (CPU is fine) or locally on Python 3.10+.

## 0. Install

In [ ]:
# In Colab, uncomment the next line (gensim, pyLDAvis and wordcloud are not preinstalled)
# !pip install -q "gensim>=4.4" "pyLDAvis>=3.4" wordcloud

In [ ]:
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gensim
from gensim.corpora import Dictionary
from gensim.models import LdaModel, Phrases, CoherenceModel
from gensim.models.phrases import ENGLISH_CONNECTOR_WORDS
from gensim.parsing.preprocessing import STOPWORDS
from gensim.utils import simple_preprocess

warnings.filterwarnings("ignore", category=DeprecationWarning)
print("gensim", gensim.__version__)
RANDOM_STATE = 42

## 1. Load your content

**Your own site:** crawl it (Screaming Frog, Sitebulb, or any crawler), export one row per page with the URL and the main body text (in Screaming Frog, use *Custom Extraction* on your content container), and save it as CSV. Set `USE_DEMO = False` and point `INPUT_CSV`, `URL_COL`, `TEXT_COL` and (optionally) `TITLE_COL` at it.

**Demo:** if you just want to see it work, leave `USE_DEMO = True`. It uses six groups from the public 20 Newsgroups dataset (cars, space, graphics, guns politics, hockey, medicine), around 3,500 user-written posts. Each post has a long body *and* a short subject line, which is perfect for the short-text experiment in section 10.

In [ ]:
USE_DEMO = True

INPUT_CSV = "crawl_export.csv"   # your file
URL_COL   = "Address"             # Screaming Frog's URL column
TEXT_COL  = "Body Text"           # your custom-extraction column
TITLE_COL = "Title 1"             # optional, used for the short-text experiment

if USE_DEMO:
    from sklearn.datasets import fetch_20newsgroups
    cats = ["rec.autos", "sci.space", "comp.graphics", "talk.politics.guns", "rec.sport.hockey", "sci.med"]
    raw = fetch_20newsgroups(subset="train", categories=cats, random_state=1)

    def split_post(post):
        head, _, body = post.partition("\n\n")
        m = re.search(r"^Subject:\s*(.*)$", head, flags=re.M)
        subject = re.sub(r"^(re:\s*)+", "", m.group(1).strip(), flags=re.I) if m else ""
        body = "\n".join(l for l in body.splitlines()
                         if not l.lstrip().startswith((">", "|")) and not re.search(r"writes:|wrote:", l))  # drop quoted replies
        return subject, body

    rows = [split_post(p) for p in raw.data]
    df = pd.DataFrame({
        "url":   [f"demo-{i}" for i in range(len(rows))],
        "title": [s for s, _ in rows],
        "text":  [b for _, b in rows],
        "label": [raw.target_names[t] for t in raw.target],   # only used to sanity-check topics
    })
else:
    src = pd.read_csv(INPUT_CSV)
    df = pd.DataFrame({
        "url":   src[URL_COL],
        "title": src[TITLE_COL] if TITLE_COL in src else "",
        "text":  src[TEXT_COL].fillna(""),
    })

df = df[df["text"].str.split().str.len() >= 20].reset_index(drop=True)   # drop near-empty pages
print(len(df), "documents")
df.head()

## 2. Clean and tokenise

LDA is a bag-of-words model, so pre-processing does a lot of the heavy lifting. Three things matter most:

* **Stopwords**: gensim's list plus your own. Add your brand name, boilerplate (cookie banners, nav labels) and anything that appears on every page.
* **Bigrams**: so "internal linking" or "search console" stay one token.
* **Topic-general words**: words that appear in a large share of documents carry little topical signal and blur topics together. `filter_extremes(no_above=0.5)` drops any word found in more than half of the documents, and `no_below=5` drops words seen in fewer than five.

In [ ]:
CUSTOM_STOPWORDS = {
    # add your brand, boilerplate and site-wide words here
    "edu", "com", "would", "like", "know", "think", "people", "get", "also", "use", "one", "writes", "article",
}
STOP = STOPWORDS.union(CUSTOM_STOPWORDS)

def tokenise(text):
    text = re.sub(r"\S+@\S+|https?://\S+|www\.\S+", " ", str(text))   # emails and URLs
    return [t for t in simple_preprocess(text, deacc=True, min_len=3) if t not in STOP and not t.isdigit()]

tokens = [tokenise(t) for t in df["text"]]

bigram = Phrases(tokens, min_count=5, threshold=20, connector_words=ENGLISH_CONNECTOR_WORDS)
tokens = [bigram[doc] for doc in tokens]

dictionary = Dictionary(tokens)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(doc) for doc in tokens]
print(f"{len(dictionary):,} unique tokens after filtering; avg {np.mean([len(d) for d in tokens]):.0f} tokens per document")

## 3. Word cloud of the whole corpus
A top-level sense of the dataset before any modelling.

In [ ]:
from wordcloud import WordCloud
freqs = {dictionary[i]: c for i, c in dictionary.cfs.items()}
wc = WordCloud(width=1200, height=500, background_color="white", colormap="Purples", max_words=150, random_state=RANDOM_STATE)
plt.figure(figsize=(12, 5)); plt.imshow(wc.generate_from_frequencies(freqs)); plt.axis("off"); plt.show()

## 4. Train the LDA model

gensim trains LDA with **online variational Bayes** (Hoffman, Blei & Bach, 2010), a scalable descendant of the variational EM approach in the original 2003 paper. If you specifically want Gibbs sampling, look at `tomotopy`.

* `num_topics`: how many topics you want (you must decide this up front; section 5 helps).
* `alpha`: the document-to-topic prior. `"auto"` lets gensim learn an asymmetric prior from the data.
* `eta` (often called beta): the topic-to-word prior. `"auto"` learns it too.

In [ ]:
NUM_TOPICS = 6

def train_lda(corpus, k, alpha="auto", eta="auto", passes=10):
    return LdaModel(corpus=corpus, id2word=dictionary, num_topics=k, alpha=alpha, eta=eta,
                    passes=passes, iterations=200, chunksize=2000, random_state=RANDOM_STATE, eval_every=None)

lda = train_lda(corpus, NUM_TOPICS)
for t in range(NUM_TOPICS):
    print(t, ", ".join(w for w, _ in lda.show_topic(t, topn=10)))

## 5. Is it any good? Perplexity and coherence

* **Perplexity**: how well the model predicts unseen documents. *Lower is better.* Careful: gensim's `log_perplexity()` returns a per-word log-likelihood **bound** (a negative number), not the perplexity itself. Perplexity is `2 ** (-bound)`. Comparing raw bounds across models is a classic mix-up.
* **Coherence (c_v)**: how semantically related the top words in each topic are, roughly "does this make sense to a human?". *Higher is better*, on a 0 to 1 scale. Research (Chang et al., 2009) found that perplexity and human judgement often disagree, so I lean on coherence when choosing a model.

In [ ]:
def coherence(model_or_topics, texts=tokens, dictionary=dictionary, topn=10):
    if isinstance(model_or_topics, list):
        cm = CoherenceModel(topics=model_or_topics, texts=texts, dictionary=dictionary, coherence="c_v", topn=topn, processes=1)
    else:
        cm = CoherenceModel(model=model_or_topics, texts=texts, dictionary=dictionary, coherence="c_v", topn=topn, processes=1)
    return cm.get_coherence()

# hold out 10% of documents to measure perplexity on unseen text
rng = np.random.default_rng(RANDOM_STATE)
holdout_idx = set(rng.choice(len(corpus), size=len(corpus) // 10, replace=False))
train_corpus = [d for i, d in enumerate(corpus) if i not in holdout_idx]
holdout_corpus = [d for i, d in enumerate(corpus) if i in holdout_idx]

def perplexity(model, held=holdout_corpus):
    return 2 ** (-model.log_perplexity(held))

print(f"Perplexity: {perplexity(lda):,.0f}   Coherence (c_v): {coherence(lda):.3f}")

### Tune the number of topics, alpha and eta
Too high an alpha spreads every document across many topics and blurs them; too high an eta makes topics broad and generic. Test a few configurations, pick the highest coherence, rebuild with those settings.

In [ ]:
results = []
for k in [4, 6, 8, 10, 12]:
    m = train_lda(train_corpus, k, passes=5)
    results.append({"num_topics": k, "alpha": "auto", "eta": "auto", "coherence": coherence(m), "perplexity": perplexity(m)})
best_k = max(results, key=lambda r: r["coherence"])["num_topics"]

for alpha in ["symmetric", "asymmetric", 0.1]:
    for eta in ["auto", 0.01]:
        m = train_lda(train_corpus, best_k, alpha=alpha, eta=eta, passes=5)
        results.append({"num_topics": best_k, "alpha": alpha, "eta": eta, "coherence": coherence(m), "perplexity": perplexity(m)})

tuning = pd.DataFrame(results).sort_values("coherence", ascending=False)
tuning.round(3)

In [ ]:
best = tuning.iloc[0]
NUM_TOPICS = int(best["num_topics"])
lda = train_lda(corpus, NUM_TOPICS, alpha=best["alpha"], eta=best["eta"])
print(f"Final model: {NUM_TOPICS} topics, alpha={best['alpha']}, eta={best['eta']}, coherence={coherence(lda):.3f}")
topic_words = pd.DataFrame({f"topic_{t}": [w for w, _ in lda.show_topic(t, topn=15)] for t in range(NUM_TOPICS)})
topic_words

## 6. The pyLDAvis map
Left: each topic is a bubble, sized by its share of the corpus and positioned by similarity to the others. Click a bubble to see its 30 most relevant terms on the right. Slide λ (lambda) towards 0 to see the words most *distinctive* to a topic rather than the most frequent.

In [ ]:
import pyLDAvis, pyLDAvis.gensim_models
vis = pyLDAvis.gensim_models.prepare(lda, corpus, dictionary, mds="mmds", sort_topics=False)  # "mmds" avoids a complex-number bug in the default PCoA on recent NumPy/SciPy
pyLDAvis.save_html(vis, "lda_pyldavis.html")
print("Saved lda_pyldavis.html (open it in a browser). In Colab, run pyLDAvis.enable_notebook() and then display `vis`.")

## 7. Which pages belong to which topic?
LDA is a **soft** clustering method: every page gets a probability for every topic. That is what makes it good at surfacing the several subtopics inside one page.

In [ ]:
doc_topic = np.zeros((len(corpus), NUM_TOPICS))
for i, bow in enumerate(corpus):
    for t, p in lda.get_document_topics(bow, minimum_probability=0.0):
        doc_topic[i, t] = p

topic_labels = {t: "_".join(w for w, _ in lda.show_topic(t, topn=3)) for t in range(NUM_TOPICS)}
probs = pd.DataFrame(doc_topic, columns=[f"{t}_{topic_labels[t]}" for t in range(NUM_TOPICS)])
pages = pd.concat([df[["url", "title"]], probs], axis=1)
pages["dominant_topic"] = doc_topic.argmax(axis=1)
pages["dominant_prob"] = doc_topic.max(axis=1)
pages["n_topics_over_20pct"] = (doc_topic > 0.2).sum(axis=1)
pages.head()

In [ ]:
if "label" in df:   # demo only: do the topics line up with the original groups?
    display(pd.crosstab(df["label"], pages["dominant_topic"].map(topic_labels)))

In [ ]:
# Top 5 pages per topic, ranked by probability: the input for a topic-based internal-linking exercise
top_pages = (pages.sort_values("dominant_prob", ascending=False)
                  .groupby("dominant_topic").head(5)
                  .sort_values(["dominant_topic", "dominant_prob"], ascending=[True, False]))
top_pages[["dominant_topic", "url", "title", "dominant_prob"]].assign(topic=lambda d: d.dominant_topic.map(topic_labels))

## 8. Topic-to-topic correlations and internal-link candidates

The no-code jsLDA app gives you a topic correlation view: which topics tend to show up in the same documents. Here is the same idea in code, plus a list of link candidates per page, based on how similar two pages' topic mixes are (Hellinger distance, the standard way to compare probability distributions).

In [ ]:
corr = pd.DataFrame(np.corrcoef(doc_topic.T), index=topic_labels.values(), columns=topic_labels.values())
plt.figure(figsize=(7, 6)); plt.imshow(corr, cmap="PuOr", vmin=-1, vmax=1); plt.colorbar(label="correlation")
plt.xticks(range(NUM_TOPICS), corr.columns, rotation=60, ha="right"); plt.yticks(range(NUM_TOPICS), corr.index)
plt.title("Topic-to-topic correlation across documents"); plt.tight_layout(); plt.show()

In [ ]:
def hellinger_matrix(P):
    S = np.sqrt(P)
    # ||sqrt(p) - sqrt(q)||^2 = 2 - 2 * sqrt(p).sqrt(q)
    return np.sqrt(np.clip(1 - S @ S.T, 0, None))

TOP_N_LINKS = 5
H = hellinger_matrix(doc_topic)
np.fill_diagonal(H, np.inf)
link_rows = []
for i in range(len(df)):
    for j in np.argsort(H[i])[:TOP_N_LINKS]:
        link_rows.append({"source_url": df.at[i, "url"], "source_title": df.at[i, "title"],
                          "target_url": df.at[j, "url"], "target_title": df.at[j, "title"],
                          "shared_topic": topic_labels[int(np.argmax(np.minimum(doc_topic[i], doc_topic[j])))],
                          "topic_similarity": round(1 - float(H[i, j]), 3)})
links = pd.DataFrame(link_rows)
links.head(10)

Treat this as a **shortlist**, not a decision. A human still checks that each link genuinely helps the reader, and that the anchor text makes sense in context.

## 9. NMF as a comparison model
Non-negative Matrix Factorisation on TF-IDF is fast, deterministic-ish and often produces cleaner topics than LDA on shorter texts. We score it with the same c_v coherence so the comparison is fair.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

def nmf_topics(token_lists, k, topn=10):
    vec = TfidfVectorizer(analyzer=lambda doc: [t for t in doc if t in dictionary.token2id])
    X = vec.fit_transform(token_lists)
    nmf = NMF(n_components=k, init="nndsvda", random_state=RANDOM_STATE, max_iter=500).fit(X)
    vocab = vec.get_feature_names_out()
    return [[vocab[i] for i in comp.argsort()[::-1][:topn]] for comp in nmf.components_], nmf.transform(X)

nmf_words, nmf_doc_topic = nmf_topics(tokens, NUM_TOPICS)
for t, words in enumerate(nmf_words):
    print(t, ", ".join(words))
print(f"\nCoherence (c_v)  LDA: {coherence(lda):.3f}   NMF: {coherence(nmf_words):.3f}")

## 10. The short-text experiment
Same documents, same pipeline, same number of topics: once on the full text, once on the **title / subject line only**. Watch coherence drop and the topics get muddier. This is the data-sparsity problem in action: a five-word title gives LDA almost no co-occurrence evidence to work with.

In [ ]:
if df["title"].astype(str).str.len().sum() > 0:
    short_tokens = [bigram[tokenise(t)] for t in df["title"]]
    # duplicate titles (thread replies here, duplicate page titles on a site) would inflate the results, so keep the first of each
    seen, keep = set(), []
    for i, (t, d) in enumerate(zip(df["title"], short_tokens)):
        if len(d) >= 2 and str(t).lower().strip() not in seen:
            seen.add(str(t).lower().strip()); keep.append(i)
    short_tokens = [short_tokens[i] for i in keep]
    short_dict = Dictionary(short_tokens); short_dict.filter_extremes(no_below=3, no_above=0.5)
    short_corpus = [short_dict.doc2bow(d) for d in short_tokens]
    lda_short = LdaModel(corpus=short_corpus, id2word=short_dict, num_topics=NUM_TOPICS, alpha="auto", eta="auto",
                         passes=10, iterations=200, random_state=RANDOM_STATE, eval_every=None)
    print(f"Average tokens per document  full text: {np.mean([len(d) for d in tokens]):.0f}   titles: {np.mean([len(d) for d in short_tokens]):.1f}\n")
    for t in range(NUM_TOPICS):
        print("titles LDA", t, ", ".join(w for w, _ in lda_short.show_topic(t, topn=8)))
    # score every model against the same reference texts (the full-text tokens) so the numbers are comparable
    full_ref = [tokens[i] for i in keep]
    ref_dict = Dictionary(full_ref)
    def ref_cv(words):   # score top words against the same reference texts (full text of the kept documents)
        words = [[w for w in ws if w in ref_dict.token2id][:10] for ws in words]
        return coherence([ws for ws in words if len(ws) >= 3], texts=full_ref, dictionary=ref_dict)
    lda_full_words = [[w for w, _ in lda.show_topic(t, topn=15)] for t in range(NUM_TOPICS)]
    lda_short_words = [[w for w, _ in lda_short.show_topic(t, topn=15)] for t in range(NUM_TOPICS)]
    comparison = pd.DataFrame([
        {"model": "LDA, full text", "coherence_c_v": ref_cv(lda_full_words)},
        {"model": "NMF, full text", "coherence_c_v": ref_cv(nmf_topics(tokens, NUM_TOPICS, topn=15)[0])},
        {"model": "LDA, titles only", "coherence_c_v": ref_cv(lda_short_words)},
        {"model": "NMF, titles only", "coherence_c_v": ref_cv(nmf_topics(short_tokens, NUM_TOPICS, topn=15)[0])},
    ])
    display(comparison.round(3))
else:
    print("No titles in your data, skipping the short-text experiment.")

## 11. Export
Three CSVs: page-level topic probabilities, topic keywords, and internal-link candidates. In Colab they download automatically.

In [ ]:
pages.to_csv("lda_page_topics.csv", index=False)
topic_words.to_csv("lda_topic_keywords.csv", index=False)
links.to_csv("lda_internal_link_candidates.csv", index=False)
tuning.to_csv("lda_tuning_results.csv", index=False)
try:
    from google.colab import files
    for f in ["lda_page_topics.csv", "lda_topic_keywords.csv", "lda_internal_link_candidates.csv", "lda_pyldavis.html"]:
        files.download(f)
except ImportError:
    print("Saved: lda_page_topics.csv, lda_topic_keywords.csv, lda_internal_link_candidates.csv, lda_tuning_results.csv, lda_pyldavis.html")